In [ ]:
# Root dataset directory on Kaggle
BASE_PATH = "/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23"

# Subfolder containing real/original videos
REAL_PATH = BASE_PATH + "/original"
# Subfolder containing manipulated/deepfake videos
FAKE_PATH = BASE_PATH + "/Deepfakes"



In [ ]:
import cv2
import os

# Extract sampled frames from one video and save them as images
def extract_frames(video_path, output_dir, label, fps=1, max_frames=20):
    # Open the video file for reading
    cap = cv2.VideoCapture(video_path)
    
    # Read original FPS to compute sampling interval
    video_fps = cap.get(cv2.CAP_PROP_FPS)
    # Fallback interval if FPS is unavailable
    interval = int(video_fps // fps) if video_fps > 0 else 25

    # count = all frames seen, saved = frames written to disk
    count, saved = 0, 0
    # Ensure destination folder exists
    os.makedirs(output_dir, exist_ok=True)

    # Keep reading frames until video ends or max_frames is reached
    while cap.isOpened() and saved < max_frames:
        ret, frame = cap.read()
        if not ret:
            break

        # Save one frame every `interval` frames
        if count % interval == 0:
            # Resize frames to model input size for CNN training
            frame = cv2.resize(frame, (128, 128))
            # Build a label-aware frame filename
            name = f"{label}_{os.path.basename(video_path)}_{saved}.jpg"
            # Write frame image to disk
            cv2.imwrite(os.path.join(output_dir, name), frame)
            saved += 1

        count += 1

    # Release video handle to free resources
    cap.release()


# Walk through nested directories and process mp4 videos
def process_recursive(input_dir, output_dir, label, limit=120):
    processed = 0
    for root, _, files in os.walk(input_dir):
        for f in files:
            if f.endswith(".mp4"):
                # Extract frames from each video file
                extract_frames(os.path.join(root, f), output_dir, label)
                processed += 1
                # Stop once the requested number of videos is reached
                if processed >= limit:
                    return

In [ ]:
# Output folder for extracted REAL frames
OUT_REAL = "/kaggle/working/dataset/deepfake/real"
# Output folder for extracted FAKE frames
OUT_FAKE = "/kaggle/working/dataset/deepfake/fake"

# Extract frames from a subset of real videos
process_recursive(REAL_PATH, OUT_REAL, "real", limit=120)
# Extract frames from a subset of fake videos
process_recursive(FAKE_PATH, OUT_FAKE, "fake", limit=120)

In [6]:
print("Real:", len(os.listdir(OUT_REAL)))
print("Fake:", len(os.listdir(OUT_FAKE)))

Real: 2015
Fake: 2015


In [ ]:
import os, shutil
from sklearn.model_selection import train_test_split

# Split one class folder into train/val/test and copy files to target structure
def split_class(src_dir, base_out):
    # List all frame images for this class
    files = os.listdir(src_dir)

    # First split: 70% train, 30% temp
    train, temp = train_test_split(files, test_size=0.3, random_state=42)
    # Second split: temp -> 15% val, 15% test
    val, test = train_test_split(temp, test_size=0.5, random_state=42)

    # Copy each split into ImageFolder-compatible directories
    for split, data in zip(["train","val","test"], [train, val, test]):
        dst = os.path.join(base_out, split, os.path.basename(src_dir))
        os.makedirs(dst, exist_ok=True)
        for f in data:
            shutil.copy(os.path.join(src_dir, f), os.path.join(dst, f))


# Root output folder for final train/val/test dataset
BASE_OUT = "/kaggle/working/final_dataset"

# Create train/val/test splits for real class
split_class(OUT_REAL, BASE_OUT)
# Create train/val/test splits for fake class
split_class(OUT_FAKE, BASE_OUT)

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Convert images to PyTorch tensors
transform = transforms.Compose([
    transforms.ToTensor()
])

# Build datasets from folder structure: train/val/test with class subfolders
train_data = datasets.ImageFolder(BASE_OUT + "/train", transform=transform)
val_data   = datasets.ImageFolder(BASE_OUT + "/val", transform=transform)
test_data  = datasets.ImageFolder(BASE_OUT + "/test", transform=transform)

# DataLoader for mini-batch training
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
# DataLoader for validation
val_loader   = DataLoader(val_data, batch_size=32)
# DataLoader for final testing
test_loader  = DataLoader(test_data, batch_size=32)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Baseline CNN for binary classification (real vs fake)
class BaselineCNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Feature extractor with two convolution blocks
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool  = nn.MaxPool2d(2, 2)
        
        # Classifier head after flattening feature maps
        self.fc1 = nn.Linear(64 * 32 * 32, 128)
        self.fc2 = nn.Linear(128, 2)

    def forward(self, x):
        # Block 1: conv + relu + downsample
        x = self.pool(F.relu(self.conv1(x)))   # 128 → 64
        # Block 2: conv + relu + downsample
        x = self.pool(F.relu(self.conv2(x)))   # 64 → 32
        
        # Flatten spatial features into a vector
        x = x.view(x.size(0), -1)
        # Dense layer for learned representation
        x = F.relu(self.fc1(x))
        # Output logits for 2 classes
        x = self.fc2(x)
        return x

In [ ]:
# Select GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize baseline model and move it to target device
model = BaselineCNN().to(device)
# Adam optimizer for parameter updates
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
# Cross-entropy loss for 2-class classification
criterion = nn.CrossEntropyLoss()

# Train baseline model for 5 epochs
for epoch in range(5):
    # Enable training mode (dropout/batchnorm behavior)
    model.train()
    
    # Iterate over mini-batches
    for images, labels in train_loader:
        # Move batch to device
        images, labels = images.to(device), labels.to(device)

        # Clear old gradients
        optimizer.zero_grad()
        # Forward pass: compute logits
        outputs = model(images)
        # Compute classification loss
        loss = criterion(outputs, labels)
        # Backpropagate gradients
        loss.backward()
        # Update model weights
        optimizer.step()

    print(f"Epoch {epoch} done")

Epoch 0 done
Epoch 1 done
Epoch 2 done
Epoch 3 done
Epoch 4 done


In [ ]:
# Switch to evaluation mode
model.eval()
# Counters for accuracy computation
correct, total = 0, 0

# Disable gradient tracking for faster inference
with torch.no_grad():
    for images, labels in val_loader:
        # Move validation batch to device
        images, labels = images.to(device), labels.to(device)
        # Forward pass on validation data
        outputs = model(images)
        # Convert logits to predicted class index
        _, preds = torch.max(outputs, 1)

        # Update totals for accuracy
        total += labels.size(0)
        correct += (preds == labels).sum().item()

print("Validation Accuracy:", correct / total)

Validation Accuracy: 0.9387417218543046


In [12]:
# Improved CNN(fine tuning)

In [ ]:
from torchvision import transforms

# Data augmentation to improve generalization during training
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor()
])

# Validation/test transform without augmentation
val_transform = transforms.Compose([
    transforms.ToTensor()
])

# Reload datasets with augmentation-aware transforms
train_data = datasets.ImageFolder(BASE_OUT + "/train", transform=train_transform)
val_data   = datasets.ImageFolder(BASE_OUT + "/val", transform=val_transform)
test_data  = datasets.ImageFolder(BASE_OUT + "/test", transform=val_transform)

# Rebuild loaders with updated datasets
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=32)

In [ ]:
# Deeper CNN with dropout for stronger feature learning
class ImprovedCNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Three convolution layers for richer hierarchical features
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)

        # Shared downsampling and regularization
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.5)

        # Fully connected classifier head
        self.fc1 = nn.Linear(128 * 16 * 16, 256)
        self.fc2 = nn.Linear(256, 2)

    def forward(self, x):
        # Conv block 1
        x = self.pool(F.relu(self.conv1(x)))   # 128 → 64
        # Conv block 2
        x = self.pool(F.relu(self.conv2(x)))   # 64 → 32
        # Conv block 3
        x = self.pool(F.relu(self.conv3(x)))   # 32 → 16
        
        # Flatten before dense layers
        x = x.view(x.size(0), -1)
        # Apply dropout-regularized dense layer
        x = self.dropout(F.relu(self.fc1(x)))
        # Output logits for two classes
        x = self.fc2(x)
        return x

In [ ]:
# Initialize improved CNN model
model2 = ImprovedCNN().to(device)

# Adam with lower LR and weight decay for regularization
optimizer = torch.optim.Adam(model2.parameters(), lr=0.0005, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

# Train improved model for 7 epochs
for epoch in range(7):
    model2.train()
    
    for images, labels in train_loader:
        # Move data batch to device
        images, labels = images.to(device), labels.to(device)

        # Reset gradients before backpropagation
        optimizer.zero_grad()
        # Forward pass
        outputs = model2(images)
        # Compute loss
        loss = criterion(outputs, labels)
        # Backward pass
        loss.backward()
        # Update weights
        optimizer.step()

    print(f"[ImprovedCNN] Epoch {epoch} done")

[ImprovedCNN] Epoch 0 done
[ImprovedCNN] Epoch 1 done
[ImprovedCNN] Epoch 2 done
[ImprovedCNN] Epoch 3 done
[ImprovedCNN] Epoch 4 done
[ImprovedCNN] Epoch 5 done
[ImprovedCNN] Epoch 6 done


In [ ]:
# Evaluate improved CNN on validation set
model2.eval()
correct, total = 0, 0

# Inference mode: no gradient computation
with torch.no_grad():
    for images, labels in val_loader:
        # Move validation batch to device
        images, labels = images.to(device), labels.to(device)
        # Predict class logits
        outputs = model2(images)
        # Select highest-scoring class
        _, preds = torch.max(outputs, 1)

        # Accumulate correct predictions
        total += labels.size(0)
        correct += (preds == labels).sum().item()

print("Improved CNN Validation Accuracy:", correct / total)

Improved CNN Validation Accuracy: 0.8857615894039735


In [ ]:
# fine tuned improved CNN

In [ ]:
# Resize to 224x224 for ResNet18 input requirements
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

# Validation transform for ResNet without augmentation
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [ ]:
# Rebuild datasets with ResNet-sized transforms
train_data = datasets.ImageFolder(BASE_OUT + "/train", transform=train_transform)
val_data   = datasets.ImageFolder(BASE_OUT + "/val", transform=val_transform)

# Create loaders for ResNet training and validation
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=32)

In [ ]:
from torchvision import models
from torchvision.models import ResNet18_Weights

# Load pretrained ResNet18 weights
weights = ResNet18_Weights.DEFAULT
model3 = models.resnet18(weights=weights)

# Replace final classifier for binary output
model3.fc = nn.Linear(model3.fc.in_features, 2)
# Move model to selected device
model3 = model3.to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 190MB/s]


In [ ]:
# Freeze all pretrained backbone layers first
for param in model3.parameters():
    param.requires_grad = False

# Unfreeze only the final classification layer for warm-up training
for param in model3.fc.parameters():
    param.requires_grad = True

In [ ]:
# Optimizer for initial stage (classifier-focused training)
optimizer = torch.optim.Adam(model3.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

# Stage 1: train with backbone frozen
for epoch in range(3):
    model3.train()
    
    for images, labels in train_loader:
        # Move batch to device
        images, labels = images.to(device), labels.to(device)

        # Clear gradients
        optimizer.zero_grad()
        # Forward pass through ResNet
        outputs = model3(images)
        # Compute loss against labels
        loss = criterion(outputs, labels)
        # Backpropagate
        loss.backward()
        # Apply optimizer step
        optimizer.step()

    print(f"[ResNet18 - Frozen] Epoch {epoch} done")

[ResNet18 - Frozen] Epoch 0 done
[ResNet18 - Frozen] Epoch 1 done
[ResNet18 - Frozen] Epoch 2 done


In [ ]:
# Unfreeze last residual block for deeper fine-tuning
for param in model3.layer4.parameters():
    param.requires_grad = True

In [ ]:
# Lower LR for stable fine-tuning of unfrozen layers
optimizer = torch.optim.Adam(model3.parameters(), lr=1e-5)

# Stage 2: fine-tune with layer4 unfrozen
for epoch in range(3):
    model3.train()
    
    for images, labels in train_loader:
        # Move mini-batch to device
        images, labels = images.to(device), labels.to(device)

        # Zero out previous gradients
        optimizer.zero_grad()
        # Forward pass
        outputs = model3(images)
        # Compute classification loss
        loss = criterion(outputs, labels)
        # Backward pass
        loss.backward()
        # Update trainable parameters
        optimizer.step()

    print(f"[ResNet18 - Finetune] Epoch {epoch} done")

[ResNet18 - Finetune] Epoch 0 done
[ResNet18 - Finetune] Epoch 1 done
[ResNet18 - Finetune] Epoch 2 done


In [ ]:
# Evaluate fine-tuned ResNet18 on validation set
model3.eval()
correct, total = 0, 0

# Disable grads for evaluation speed and memory savings
with torch.no_grad():
    for images, labels in val_loader:
        # Move validation data to device
        images, labels = images.to(device), labels.to(device)
        # Get prediction logits
        outputs = model3(images)
        # Choose predicted class
        _, preds = torch.max(outputs, 1)

        # Update running accuracy stats
        total += labels.size(0)
        correct += (preds == labels).sum().item()

print("ResNet18 Validation Accuracy:", correct / total)

ResNet18 Validation Accuracy: 0.9486754966887417


In [ ]:
# Save trained baseline model weights
torch.save(model.state_dict(), "/kaggle/working/baseline.pth")
# Save trained improved CNN weights
torch.save(model2.state_dict(), "/kaggle/working/improved.pth")
# Save trained ResNet18 fine-tuned weights
torch.save(model3.state_dict(), "/kaggle/working/resnet18.pth")

In [ ]:
import json
# Save class index order so inference maps outputs to labels correctly
with open("/kaggle/working/classes.json", "w") as f:
    json.dump(train_data.classes, f)